# Analysis - Availability Windows
Analysing the output from the Availability Window Experiments.

In [1]:
import pandas as pd
import pickle
import json
import os
import datetime as dt
from matplotlib import pyplot as plt
import re
import numpy as np

In [2]:
vwt_out = "D:/lco/custom_scheduler/output_files/vwt"
vwt_in = "D:/lco/custom_scheduler/input_files/vwt"
baseline_out = "D:/lco/custom_scheduler/output_files/baseline"
print(os.path.isdir(vwt_out))
print(os.path.isdir(vwt_in))
print(os.path.isdir(baseline_out))

True
True
True


In [6]:
filemap = {}
for dirname, folders, filenames in os.walk(vwt_in):
    for filename in filenames:
        output_filepath = os.path.join(vwt_out, filename)
        forename, ext = os.path.splitext(filename)
        perfect_filepath = os.path.join(vwt_out, forename + "_perfect" + ext)
        baseline_filepath = os.path.join(baseline_out, forename.replace("vwt_", "baseline_")+ext)
        print(filename)
        print(os.path.isfile(output_filepath), output_filepath)
        print(os.path.isfile(perfect_filepath), perfect_filepath)
        print(os.path.isfile(baseline_filepath), baseline_filepath)
        filemap[filename] = {
            "input": os.path.join(dirname, filename),
            "output": output_filepath,
            "perfect": perfect_filepath,
            "baseline": baseline_filepath
        }

vwt_2020-08.pkl
True D:/lco/custom_scheduler/output_files/vwt\vwt_2020-08.pkl
True D:/lco/custom_scheduler/output_files/vwt\vwt_2020-08_perfect.pkl
True D:/lco/custom_scheduler/output_files/baseline\baseline_2020-08.pkl
vwt_2021-02.pkl
True D:/lco/custom_scheduler/output_files/vwt\vwt_2021-02.pkl
True D:/lco/custom_scheduler/output_files/vwt\vwt_2021-02_perfect.pkl
True D:/lco/custom_scheduler/output_files/baseline\baseline_2021-02.pkl
vwt_2021-08.pkl
True D:/lco/custom_scheduler/output_files/vwt\vwt_2021-08.pkl
True D:/lco/custom_scheduler/output_files/vwt\vwt_2021-08_perfect.pkl
True D:/lco/custom_scheduler/output_files/baseline\baseline_2021-08.pkl
vwt_2022-02.pkl
True D:/lco/custom_scheduler/output_files/vwt\vwt_2022-02.pkl
True D:/lco/custom_scheduler/output_files/vwt\vwt_2022-02_perfect.pkl
True D:/lco/custom_scheduler/output_files/baseline\baseline_2022-02.pkl


In [7]:
calib_proposals = [
    "OGG_calib",
    "MuSCAT Commissioning",
    "auto_focus",
    "LCOEngineering",
    "COJ_calib",
    "FLOYDS standards",
    "Photometric standards",
    "standard"
]

In [13]:
for filename in filemap:
    i = pickle.load(open(filemap[filename]["input"], "rb")) #input
    o = pickle.load(open(filemap[filename]["output"], "rb")) #output
    p = pickle.load(open(filemap[filename]["perfect"], "rb")) #perfect
    b = pickle.load(open(filemap[filename]["baseline"], "rb")) #baseline

    # proposals = i["proposals"]
    # for calib_name in calib_proposals:
    #     if calib_name in proposals:
    #         proposals[calib_name] = 0

    data = i["all_requests"][["id", "new_window_length"]]
    data["scheduled"] = data["id"].isin(o["final_completed_requests"].keys())
    data["baseline"] = data["id"].isin(b["final_completed_requests"].keys())
    data["perfect"] = data["id"].isin(p["scheduled"].keys())
            
    print("FILENAME:", filename)
    for availability, group in data.groupby("new_window_length"):
        print("Availability Length:", availability)
        print("Num Requests:", len(group))
        print("Baseline:", group["baseline"].sum(), f"({round(group['baseline'].sum()/len(group)*100, 2)}%)")
        print("Scheduled:", group["scheduled"].sum(), f"({round(group['scheduled'].sum()/len(group)*100, 2)}%)")
        print("Perfect:", group["perfect"].sum(), f"({round(group['perfect'].sum()/len(group)*100, 2)}%)")
        print()
    print("===\n\n")


C:\Users\Foggy\AppData\Local\Temp\ipykernel_16596\3632505997.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data["scheduled"] = data["id"].isin(o["final_completed_requests"].keys())
C:\Users\Foggy\AppData\Local\Temp\ipykernel_16596\3632505997.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data["baseline"] = data["id"].isin(b["final_completed_requests"].keys())
C:\Users\Foggy\AppData\Local\Temp\ipykernel_16596\3632505997.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a s

FILENAME: vwt_2020-08.pkl
Availability Length: 1.0
Num Requests: 1828
Baseline: 1446 (79.1%)
Scheduled: 1359 (74.34%)
Perfect: 1276 (69.8%)

Availability Length: 3.0
Num Requests: 1830
Baseline: 1462 (79.89%)
Scheduled: 1416 (77.38%)
Perfect: 1344 (73.44%)

Availability Length: 7.0
Num Requests: 1830
Baseline: 1448 (79.13%)
Scheduled: 1445 (78.96%)
Perfect: 1382 (75.52%)

Availability Length: 28.0
Num Requests: 459
Baseline: 354 (77.12%)
Scheduled: 365 (79.52%)
Perfect: 363 (79.08%)

===




C:\Users\Foggy\AppData\Local\Temp\ipykernel_16596\3632505997.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data["scheduled"] = data["id"].isin(o["final_completed_requests"].keys())
C:\Users\Foggy\AppData\Local\Temp\ipykernel_16596\3632505997.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data["baseline"] = data["id"].isin(b["final_completed_requests"].keys())
C:\Users\Foggy\AppData\Local\Temp\ipykernel_16596\3632505997.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a s

FILENAME: vwt_2021-02.pkl
Availability Length: 1.0
Num Requests: 4008
Baseline: 1738 (43.36%)
Scheduled: 1311 (32.71%)
Perfect: 1032 (25.75%)

Availability Length: 3.0
Num Requests: 4010
Baseline: 1704 (42.49%)
Scheduled: 1604 (40.0%)
Perfect: 1208 (30.12%)

Availability Length: 7.0
Num Requests: 4009
Baseline: 1720 (42.9%)
Scheduled: 1784 (44.5%)
Perfect: 1333 (33.25%)

Availability Length: 28.0
Num Requests: 1004
Baseline: 441 (43.92%)
Scheduled: 532 (52.99%)
Perfect: 430 (42.83%)

===




C:\Users\Foggy\AppData\Local\Temp\ipykernel_16596\3632505997.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data["scheduled"] = data["id"].isin(o["final_completed_requests"].keys())
C:\Users\Foggy\AppData\Local\Temp\ipykernel_16596\3632505997.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data["baseline"] = data["id"].isin(b["final_completed_requests"].keys())
C:\Users\Foggy\AppData\Local\Temp\ipykernel_16596\3632505997.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a s

FILENAME: vwt_2021-08.pkl
Availability Length: 1.0
Num Requests: 2567
Baseline: 1979 (77.09%)
Scheduled: 1875 (73.04%)
Perfect: 1601 (62.37%)

Availability Length: 3.0
Num Requests: 2569
Baseline: 1952 (75.98%)
Scheduled: 1959 (76.26%)
Perfect: 1678 (65.32%)

Availability Length: 7.0
Num Requests: 2569
Baseline: 1933 (75.24%)
Scheduled: 2023 (78.75%)
Perfect: 1864 (72.56%)

Availability Length: 28.0
Num Requests: 644
Baseline: 480 (74.53%)
Scheduled: 534 (82.92%)
Perfect: 500 (77.64%)

===


FILENAME: vwt_2022-02.pkl
Availability Length: 1.0
Num Requests: 2607
Baseline: 2015 (77.29%)
Scheduled: 1889 (72.46%)
Perfect: 1511 (57.96%)

Availability Length: 3.0
Num Requests: 2609
Baseline: 2018 (77.35%)
Scheduled: 2013 (77.16%)
Perfect: 1634 (62.63%)

Availability Length: 7.0
Num Requests: 2609
Baseline: 2025 (77.62%)
Scheduled: 2100 (80.49%)
Perfect: 1727 (66.19%)

Availability Length: 28.0
Num Requests: 653
Baseline: 487 (74.58%)
Scheduled: 533 (81.62%)
Perfect: 478 (73.2%)

===




C:\Users\Foggy\AppData\Local\Temp\ipykernel_16596\3632505997.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data["scheduled"] = data["id"].isin(o["final_completed_requests"].keys())
C:\Users\Foggy\AppData\Local\Temp\ipykernel_16596\3632505997.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data["baseline"] = data["id"].isin(b["final_completed_requests"].keys())
C:\Users\Foggy\AppData\Local\Temp\ipykernel_16596\3632505997.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a s

In [11]:
i["all_requests"]

,id,optimization_type,total_duration,observation_state,location,availability_windows,visibility_windows,config_data,request_group_id,ipp_value,...,created,proposal_id,visibilities,tuples,valid_telescopes,old_windows,win_start,win_end,new_window_length,windows
2373081,2373081,TIME,3088,WINDOW_EXPIRED,{'telescope_class': '2m0'},"[{'start': '2021-01-30T22:03:25Z', 'end': '202...","{'coj': datetime.datetime(2021, 1, 31, 10, 2, ...","[{'config_type': 'LAMP_FLAT', 'instrument_type...",1140990,1.0,...,2021-01-31 06:03:25.754436,NOAO2020B-011,"{'coj': datetime.datetime(2021, 1, 31, 10, 2, ...","((2M0-FLOYDS-SCICAM,), , , )","{'coj': ['2m0a.clma.coj'], 'ogg': ['2m0a.clma....","{'2m0a.clma.coj': datetime.datetime(2021, 1, 3...",2021-01-31 05:04:50.762038,2021-01-31 14:19:27.806779,3.0,"{'2m0a.clma.coj': datetime.datetime(2021, 1, 3..."
2373019,2373019,TIME,19422,WINDOW_EXPIRED,{'telescope_class': '2m0'},"[{'start': '2021-01-31T05:05:00Z', 'end': '202...","{'coj': datetime.datetime(2021, 1, 31, 10, 2, ...","[{'config_type': 'REPEAT_EXPOSE', 'instrument_...",1140950,1.0,...,2021-01-31 03:04:57.140713,KEY2020B-005,"{'ogg': datetime.datetime(2021, 1, 31, 5, 5)(s...","((2M0-SCICAM-MUSCAT,), , , )","{'coj': ['2m0a.clma.coj'], 'ogg': ['2m0a.clma....","{'2m0a.clma.ogg': datetime.datetime(2021, 1, 3...",2021-01-31 05:05:00.000000,2021-01-31 10:30:00.000000,7.0,"{'2m0a.clma.coj': datetime.datetime(2021, 1, 3..."
2373018,2373018,TIME,3388,WINDOW_EXPIRED,{'telescope_class': '2m0'},"[{'start': '2021-01-30T19:02:43Z', 'end': '202...","{'ogg': datetime.datetime(2021, 1, 31, 5, 4, 5...","[{'config_type': 'LAMP_FLAT', 'instrument_type...",1140949,1.0,...,2021-01-31 03:02:43.601319,NOAO2020B-011,"{'ogg': datetime.datetime(2021, 1, 31, 5, 4, 5...","((2M0-FLOYDS-SCICAM,), , , )","{'coj': ['2m0a.clma.coj'], 'ogg': ['2m0a.clma....","{'2m0a.clma.ogg': datetime.datetime(2021, 1, 3...",2021-01-31 05:04:50.762038,2021-01-31 08:20:29.245336,7.0,"{'2m0a.clma.ogg': datetime.datetime(2021, 1, 3..."
2373014,2373014,TIME,3388,WINDOW_EXPIRED,{'telescope_class': '2m0'},"[{'start': '2021-01-30T19:01:09Z', 'end': '202...","{'ogg': datetime.datetime(2021, 1, 31, 7, 13, ...","[{'config_type': 'LAMP_FLAT', 'instrument_type...",1140947,1.0,...,2021-01-31 03:01:09.805896,NOAO2020B-011,"{'ogg': datetime.datetime(2021, 1, 31, 7, 13, ...","((2M0-FLOYDS-SCICAM,), , , )","{'coj': ['2m0a.clma.coj'], 'ogg': ['2m0a.clma....","{'2m0a.clma.ogg': datetime.datetime(2021, 1, 3...",2021-01-31 07:13:29.594408,2021-01-31 16:11:55.565044,1.0,"{'2m0a.clma.ogg': datetime.datetime(2021, 1, 3..."
2372999,2372999,TIME,720,WINDOW_EXPIRED,"{'telescope_class': '2m0', 'site': 'coj', 'enc...","[{'start': '2021-01-31T18:02:26.887954Z', 'end...","{'coj': datetime.datetime(2021, 1, 31, 18, 2, ...","[{'config_type': 'AUTO_FOCUS', 'instrument_typ...",1140932,1.05,...,2021-01-31 02:00:28.264852,auto_focus,"{'coj': datetime.datetime(2021, 1, 31, 18, 2, ...","((2M0-SCICAM-SPECTRAL,), coj, clma, 2m0a)",{'coj': ['2m0a.clma.coj']},"{'2m0a.clma.coj': datetime.datetime(2021, 1, 3...",2021-01-31 18:02:26.887954,2021-01-31 18:32:16.281317,NaN,"{'2m0a.clma.coj': datetime.datetime(2021, 1, 3..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2165155,2165155,TIME,417,WINDOW_EXPIRED,{'telescope_class': '2m0'},"[{'start': '2020-12-27T09:54:00Z', 'end': '202...","{'ogg': datetime.datetime(2020, 12, 28, 4, 44,...","[{'config_type': 'EXPOSE', 'instrument_type': ...",1011570,1.05,...,2020-06-28 21:02:10.760401,FTP2020B-003,"{'ogg': datetime.datetime(2020, 12, 28, 4, 44,...","((2M0-SCICAM-SPECTRAL,), , , )","{'coj': ['2m0a.clma.coj'], 'ogg': ['2m0a.clma....","{'2m0a.clma.ogg': datetime.datetime(2020, 12, ...",2020-12-28 04:44:52.645582,2021-01-03 08:24:50.411309,3.0,"{'2m0a.clma.ogg': datetime.datetime(2020, 12, ..."
2165156,2165156,TIME,417,WINDOW_EXPIRED,{'telescope_class': '2m0'},"[{'start': '2021-01-03T09:54:00Z', 'end': '202...","{'ogg': datetime.datetime(2021, 1, 4, 4, 48, 5...","[